Set `max_features` (vocabulary size) and `max_len` (words per sample). After that load the imbd data into `x_train`, `y_train` and `x_test`, `y_test` and pad all the sample to the same length.

In [3]:
from tensorflow.keras.datasets import imdb
from tensorflow.keras.utils import pad_sequences 

max_features = 10000
max_len = 250

(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=max_features) 

x_train = pad_sequences(x_train, maxlen=max_len, padding='post') 
x_test = pad_sequences(x_test, maxlen=max_len, padding='post')

This code cell retrieves the word index mapping from the IMDB dataset and creates a reverse mapping to convert 
indices back to words. It then reconstructs an example review from the training data using the reverse mapping.

In [4]:
word_to_index = imdb.get_word_index()
index_to_word = dict((value, key) for (key, value) in word_to_index.items())

example_review = " ".join(index_to_word.get(i-3, "?") for i in x_train[0])
print(example_review)

? this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert ? is an amazing actor and now the same being director ? father came from the same scottish island as myself so i loved the fact there was a real connection with this film the witty remarks throughout the film were great it was just brilliant so much that i bought the film as soon as it was released for ? and would recommend it to everyone to watch and the fly fishing was amazing really cried at the end it was so sad and you know what they say if you cry at a film it must have been good and this definitely was also ? to the two little boy's that played the ? of norman and paul they were just brilliant children are often left out of the ? list i think because the stars that play them all grown up are such a big profile for the whole film but these children are amazing and should be praised for what they have done don't you thi

This custom layer combines word embeddings with positional embeddings:

1. **Token Embedding**: Converts word indices to dense vectors
2. **Position Embedding**: Adds positional information to each token

The layer creates position indices (0, 1, 2, ...) for each token position, embeds these positions into vectors, and adds them to the token embeddings.

In [5]:
from tensorflow.keras.layers import Layer, Embedding
import tensorflow as tf

class TokenAndPositionEmbedding(Layer):
    def __init__(self, seq_len, vocab_size, emb_dim):
        super(TokenAndPositionEmbedding, self).__init__()
        self.token_emb = Embedding(input_dim=vocab_size, output_dim=emb_dim)
        self.pos_emb = Embedding(input_dim=seq_len, output_dim=emb_dim)

    def call(self, x_input):
        seq_len = tf.shape(x_input)[-1]
        positions = tf.range(start=0, limit=seq_len, delta=1)
        positions = self.pos_emb(positions)
        x_input = self.token_emb(x_input)
        return x_input + positions

1. **Embedding Layer**: Processes input sequences using the custom TokenAndPositionEmbedding
2. **Multi-Head Attention**: Applies self-attention with 2 heads to capture context
3. **Residual Connection**: Adds attention output back to original input (x = Add()([attention_output, x]))
4. **Layer Normalization**: Stabilizes training by normalizing features
5. **Global Pooling & Classification**: Aggregates sequence features and predicts sentiment

In [6]:
from keras.layers import Add, LayerNormalization
from tensorflow.keras.layers import MultiHeadAttention
from tensorflow.keras.layers import Input, Dense, GlobalAveragePooling1D, Dropout
from tensorflow.keras.models import Model

embed_dim = 32

num_heads = 2
key_dim = embed_dim // num_heads

inputs = Input(shape=(max_len,)) 
x = TokenAndPositionEmbedding(max_len, max_features, embed_dim)(inputs)
attention_output = MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)(x, x)
x = Add()([attention_output, x])
x = LayerNormalization()(x)
x = GlobalAveragePooling1D()(x)
x = Dropout(0.5)(x) 
outputs = Dense(1, activation='sigmoid')(x) 
att_model = Model(inputs=inputs, outputs=outputs) 

att_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy']) 

att_model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 250)]        0           []                               
                                                                                                  
 token_and_position_embedding (  (None, 250, 32)     328000      ['input_1[0][0]']                
 TokenAndPositionEmbedding)                                                                       
                                                                                                  
 multi_head_attention (MultiHea  (None, 250, 32)     4224        ['token_and_position_embedding[0]
 dAttention)                                                     [0]',                            
                                                                  'token_and_position_embeddin

This code cell trains the model with 5 epochs and batch size of 32.

In [7]:
att_model.fit(x_train, y_train, epochs=5, batch_size=32)


Epoch 1/5


2025-04-22 09:22:28.579748: W tensorflow/tsl/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz


782/782 [==============================] - 14s 18ms/step - loss: 0.4017 - accuracy: 0.8101
Epoch 2/5
782/782 [==============================] - 14s 18ms/step - loss: 0.2256 - accuracy: 0.9119
Epoch 3/5
782/782 [==============================] - 15s 19ms/step - loss: 0.1740 - accuracy: 0.9371
Epoch 4/5
782/782 [==============================] - 15s 19ms/step - loss: 0.1379 - accuracy: 0.9533
Epoch 5/5
782/782 [==============================] - 15s 20ms/step - loss: 0.1079 - accuracy: 0.9653


This code evaluates the trained model on the test dataset and displays the accuracy. The model achieves 85.54% accuracy on the IMDB test set.

In [8]:
print(f'Test accuracy = {att_model.evaluate(x_test, y_test)[1]:.4f}')


782/782 [==============================] - 7s 9ms/step - loss: 0.4386 - accuracy: 0.8554
Test accuracy = 0.8554
